In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.preprocessing import MinMaxScaler
import joblib
import json
import os

In [2]:
# Матрица фичей — только числа для модели
data = pd.read_parquet("../data/features/feature_matrix.parquet")
# Полный датафрейм — для отчёта и объяснений
data_full = pd.read_parquet("../data/features/data_with_features.parquet")
print(f"Матрица фичей: {data.shape}")
print(f"Полный датафрейм: {data_full.shape}")

Матрица фичей: (368014, 29)
Полный датафрейм: (368014, 64)


# 1. Isolation Forest

In [3]:
model = IsolationForest(
    contamination=0.01,  # Ожидаемая доля аномалий — 1%
    n_estimators=200,    # Количество деревьев
    max_samples="auto",  # Размер подвыборки для каждого дерева
    random_state=42,
    n_jobs=-1            # Используй все ядра
)
model.fit(data)

,"n_estimators n_estimators: int, default=100The number of base estimators in the ensemble.",200
,"max_samples max_samples: ""auto"", int or float, default=""auto""The number of samples to draw from X to train each base estimator.- If int, then draw `max_samples` samples.- If float, then draw `max_samples * X.shape[0]` samples.- If ""auto"", then `max_samples=min(256, n_samples)`.If max_samples is larger than the number of samples provided,all samples will be used for all trees (no sampling).",'auto'
,"contamination contamination: 'auto' or float, default='auto'The amount of contamination of the data set, i.e. the proportionof outliers in the data set. Used when fitting to define the thresholdon the scores of the samples.- If 'auto', the threshold is determined as in the original paper.- If float, the contamination should be in the range (0, 0.5]... versionchanged:: 0.22 The default value of ``contamination`` changed from 0.1 to ``'auto'``.",0.01
,"max_features max_features: int or float, default=1.0The number of features to draw from X to train each base estimator.- If int, then draw `max_features` features.- If float, then draw `max(1, int(max_features * n_features_in_))` features.Note: using a float number less than 1.0 or integer less than number offeatures will enable feature subsampling and leads to a longer runtime.",1.0
,"bootstrap bootstrap: bool, default=FalseIf True, individual trees are fit on random subsets of the trainingdata sampled with replacement. If False, sampling without replacementis performed.",False
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel for :meth:`fit`. ``None`` means 1unless in a :obj:`joblib.parallel_backend` context. ``-1`` means usingall processors. See :term:`Glossary ` for more details.",-1
,"random_state random_state: int, RandomState instance or None, default=NoneControls the pseudo-randomness of the selection of the featureand split values for each branching step and each tree in the forest.Pass an int for reproducible results across multiple function calls.See :term:`Glossary `.",42
,"verbose verbose: int, default=0Controls the verbosity of the tree building process.",0
,"warm_start warm_start: bool, default=FalseWhen set to ``True``, reuse the solution of the previous call to fitand add more estimators to the ensemble, otherwise, just fit a wholenew forest. See :term:`the Glossary `... versionadded:: 0.21",False


In [4]:
# Считаем скоры ДО добавления в датафрейм — иначе модель упадёт при predict
scores = model.decision_function(data)  # чем ниже — тем аномальнее
labels = model.predict(data)            # 1 = норма, -1 = аномалия

data_full["anomaly_score"] = scores
data_full["anomaly_label"] = labels

n_anomalies = (labels == -1).sum()
print(f"Найдено аномалий: {n_anomalies} ({n_anomalies/len(data)*100:.2f}%)")

Найдено аномалий: 3681 (1.00%)


# 2. Local Outlier Factor

In [5]:
lof = LocalOutlierFactor(
    n_neighbors=50,      
    contamination=0.01,
    novelty=False
)
lof_labels = lof.fit_predict(data)
lof_scores = lof.negative_outlier_factor_  # чем ниже — тем аномальнее

data_full["lof_score"] = lof_scores
data_full["lof_label"] = lof_labels

n_lof = (lof_labels == -1).sum()
print(f"LOF нашёл аномалий: {n_lof} ({n_lof/len(data)*100:.2f}%)")

LOF нашёл аномалий: 3681 (1.00%)


/Users/rodion/VsCodeProjects/auditor/auditor/venv/lib/python3.13/site-packages/sklearn/neighbors/_lof.py:325: UserWarning: Duplicate values are leading to incorrect results. Increase the number of neighbors for more accurate results.
  warnings.warn(


# 3. Ансамбль

In [6]:
scaler = MinMaxScaler()

# Инвертируем скоры (чем ниже оригинал = аномальнее → после инверсии чем выше = аномальнее)
iforest_normalized = scaler.fit_transform(-data_full["anomaly_score"].values.reshape(-1, 1))
lof_normalized = scaler.fit_transform(-lof_scores.reshape(-1, 1))

# Среднее двух нормализованных скоров
data_full["ensemble_score"] = (iforest_normalized.flatten() + lof_normalized.flatten()) / 2

print(f"ensemble_score: min={data_full['ensemble_score'].min():.3f}, max={data_full['ensemble_score'].max():.3f}")

ensemble_score: min=0.000, max=0.732


# 4. Результаты — топ аномалий

In [7]:
# Whitelist — убираем заведомо нормальные типы операций
whitelist = [
    "Регламентная операция",
    "Отражение зарплаты в бухучете",
    "Операции ЕНС",
    "Уведомление об исчисленных суммах налогов",
    "Сведения об удержанном НДФЛ",
]

top40 = data_full[~data_full["ТипДокумента"].isin(whitelist)].nlargest(40, "ensemble_score")[
    ["period", "contractor", "account_dt", "account_kt", "amount",
     "ТипДокумента", "anomaly_score", "ensemble_score", "Содержание"]
]
print(top40.to_string())

                    period                                                    contractor account_dt account_kt      amount                          ТипДокумента  anomaly_score  ensemble_score                                                                                                                                              Содержание
158215 2024-11-18 16:16:48                                  Путрашевич Роман Геннадьевич      50.01      62.01     7795.20                  Поступление наличных       0.072574        0.731726                                                                                                       Реализация товаров и услуг УТ-12297 от 05.11.2024
158216 2024-11-18 16:17:09                                  Путрашевич Роман Геннадьевич      50.01      62.01    20832.00                  Поступление наличных       0.061855        0.652639                                                                                                       Реализация товаров

# 5. Сохранение

In [8]:
os.makedirs("../models", exist_ok=True)

# Сохраняем модели (pkl в .gitignore — не попадут в git)
joblib.dump(model, "../models/isolation_forest_v1.pkl")
joblib.dump(lof, "../models/lof_v1.pkl")
joblib.dump(scaler, "../models/feature_scaler_v1.pkl")

# Конфигурация (маленький json — можно в git)
config = {
    "feature_cols": data.columns.tolist(),
    "contamination": 0.01,
    "model_type": "IsolationForest+LOF ensemble",
    "n_estimators": 200,
    "lof_n_neighbors": 50,
    "training_date": str(pd.Timestamp.now()),
    "training_rows": len(data),
    "whitelist_doc_types": whitelist,
}
with open("../models/config_v1.json", "w", encoding="utf-8") as f:
    json.dump(config, f, indent=2, ensure_ascii=False)

# Сохраняем датафрейм со скорами
data_full.to_parquet("../data/features/data_with_scores.parquet", index=False)

print("Всё сохранено")

Всё сохранено
